In [ ]:
import polyase as poly
import pandas as pd
import RNApysoforms as RNApy
import polars as pl

In [ ]:
poly.__version__

# Running Polyase on tetraploid potato sample 

### Required Files



- **Syntelog finder output**: To assign genes based on syntelogous relationships
- **Oarfish directory**: Contains transcript quantification data
- **GFF file (liftoff annotation)**: Genome annotation from liftoff
- **GFF file (novel transcripts)**: Novel identified transcripts
- **Functional gene annotation**: Gene functional annotations

### Installation of polyase 


You can install polyase using pip:
```bash
pip install polyase
```


## 1) Reading in the data 

In this section we will read in the data genespace output file, transcript quanitfication directory, tx2gene. 

In [ ]:
# Metadata for genes
var_obs_file = "potato-allelic-orthogroups/output_desiree_liftoff/genespace/De_v1.unitato_liftoff_haplotap_genespace_categories.tsv"
# directory of gene counts from oarfish
gene_counts_dir = "nf-core-plantlongrnaseq/output_desiree_liftoff_phased_CDS2exon/oarfish/"
tx2gene = "nf-core-plantlongrnaseq/output_desiree_liftoff_phased_CDS2exon/custom/combined.tx2gene.tsv"
#nf-core-plantlongrnaseq/work/3f/6abd9f23b65ae2de98213bcc0ce3a5
# Define sample IDs and their conditions
sample_info = {
    "DE_C_H_13": "control",
    "DE_C_H_14": "control",
    "DE_C_H_15": "control",
    "DE_C_H_16": "control",
    "DE_H_H_17": "heat",
    "DE_H_H_18": "heat",
    "DE_H_H_19": "heat",
    "DE_H_H_20": "heat"
} 

### Dataset Information



In [ ]:
# With isoforms
adata_isoform = poly.load_ase_data(
    var_obs_file=var_obs_file,
    sample_info=sample_info,
    tx_to_gene_file=tx2gene,
    isoform_counts_dir=gene_counts_dir,
    quant_dir=gene_counts_dir
)

### Add annotation

For interpretation of results we add functional gene annotations from an additional file to the adata.var

In [ ]:
# Load functional annotation data
annotation_file = f"reference/ADAPT_liftoff/Phureja_v4-v6.1_translations.xlsx"
df_annotation = pd.read_excel(annotation_file)
df_annotation = df_annotation.iloc[:, 1:3]
df_annotation.columns = ['gene_id_unitato', 'functional_annotation']

# Remove version suffix from gene IDs (e.g., "GENE.1" -> "GENE")
adata_isoform.var['gene_id_unitato'] = adata_isoform.var['gene_id'].str.rsplit('.', n=1).str[0]
adata_isoform.var['original_index'] = adata_isoform.var.index

# Merge annotations
merged_data = pd.merge(
    adata_isoform.var,
    df_annotation,
    on='gene_id_unitato',
    how='left'
)

# Remove duplicates and restore index
merged_data = merged_data.drop_duplicates(subset=['original_index', 'gene_id'], keep='first')
merged_data.index = merged_data['original_index']
merged_data = merged_data.drop(columns=['original_index'])

# Update AnnData object
adata_isoform.var = merged_data

Let's verify whether Bambu detected any previously unannotated transcripts in our dataset.

In [ ]:
adata_isoform.var[adata_isoform.var['gene_id'].str.startswith("Bambu")].head()

### Number of isoforms per gene and novel isoforms

In [ ]:

import matplotlib.pyplot as plt
import pandas as pd

var = adata_isoform.var.copy()
var['is_novel_gene'] = var['gene_id'].str.startswith('Bambu') | var['gene_id'].str.startswith('XLOC')
var['is_novel_isoform'] = var.index.str.contains('Bambu')

# outer: unique gene counts
unique_genes = var.drop_duplicates(subset='gene_id')
novel_genes = unique_genes['is_novel_gene'].sum()
annotated_genes = (~unique_genes['is_novel_gene']).sum()

# inner: isoform counts split by gene category and isoform type
ng_novel_iso     = var[ var['is_novel_gene'] &  var['is_novel_isoform']].shape[0]
ag_novel_iso     = var[~var['is_novel_gene'] &  var['is_novel_isoform']].shape[0]
ag_annotated_iso = var[~var['is_novel_gene'] & ~var['is_novel_isoform']].shape[0]

outer_sizes  = [novel_genes, annotated_genes]
outer_colors = ["#0C3474", "#D7560C"]

inner_sizes  = [ng_novel_iso, ag_novel_iso, ag_annotated_iso]
inner_colors = ['#4C72B0', '#92b4d9', '#DD8452']  # dark/light per gene category
inner_labels = ['Novel isoform', 
                'Novel isoform', 'Annotated isoform']

fig, ax = plt.subplots(figsize=(5, 5))
# increase dpi for better resolution

total = sum(outer_sizes)
ax.pie(outer_sizes, radius=1.0,
       labels=['Novel genes', 'Annotated genes'],
       colors=outer_colors, autopct=lambda p: f'{int(round(p * total / 100))}', pctdistance=0.88,
       wedgeprops=dict(width=0.35, edgecolor='white'), startangle=90)
total = sum(inner_sizes)
ax.pie(inner_sizes, radius=0.65,
       labels=inner_labels, labeldistance=0.55,
       colors=inner_colors, autopct=lambda p: f'{int(round(p * total / 100))}', pctdistance=0.75, 
       wedgeprops=dict(width=0.35, edgecolor='white'), startangle=90)

plt.tight_layout()

# save as svg
plt.savefig("LR_DESIREE_PAPER/ANALYSIS/final_plots_svg/Novel_genes_isoforms_pie_chart.svg", format="svg")
plt.show()

Now we will aggregate transcript-level counts to gene-level counts for allele-specific expression analysis. Unique counts are just summed. For ambgious counts we take the mean, under the asseumption that multimapping between transcripts of the same gene to other genes is similar.

In [ ]:
adata_gene = poly.aggregate_transcripts_to_genes(adata_isoform)

## Check "crazy" syntelog groups

## 2) Gene-level analysis

#### Filter low expressed genes


We filter out group of syntenic genes with low expression. Each group need to have at least 20 unique reads mapped per replicate. With `library_size_dependent=True` you can also filter based on libaray size.

In [ ]:
#CPM threshold (automatically normalized)
adata_gene_filtered = adata_gene[(adata_gene.obs['condition'] == 'control'), :]
adata_gene_filtered = poly.filter_low_expressed_genes(
    adata_gene_filtered,
    min_expression=100.0,  # 20 counts
    mode= 'all',
    #library_size_dependent=True, # this filters min_expression based on CPM of unique counts
    verbose=True)

In [ ]:
adata_gene_filtered.var.groupby("synteny_category").size()

#### Calculate allelic ratios and multimapping ratios

In [ ]:
adata_isoform_filtered = poly.calculate_allelic_ratios(adata_gene_filtered , 'unique_counts')
adata_isoform_filtered = poly.calculate_allelic_ratios(adata_gene_filtered , 'em_counts')
adata_isoform_filtered = poly.calculate_multi_ratios(adata_gene_filtered, 'unique_counts', 'ambiguous_counts')

In [ ]:
adata_isoform_filtered

#### Plotting of allelic ratios for quality control

Most genes exhibit balanced allelic expression, with ratios around 0.25 (1/ploidy). Some genes display extreme ratios, which may represent biologically meaningful patterns or annotation artifacts. Applying a multimapping filter reduces noise from genes with high ambiguous read counts. We will exclude genes showing length differences, as these can introduce bias into the results as well.

In [ ]:
# Group by syntelog ID and check for equal transcript numbers across alleles
synt_ids_same_tx_number = adata_isoform_filtered.var.groupby('Synt_id').aggregate({
    'n_transcripts': set
})

# Count genes with different transcript numbers before filtering
genes_diff_tx_number = synt_ids_same_tx_number[synt_ids_same_tx_number['n_transcripts'].apply(lambda x: len(x) > 1)]
print(f"Genes with different transcript numbers across alleles: {len(genes_diff_tx_number)}")

# Keep only syntelogs where all alleles have the same number of transcripts
synt_ids_same_tx_number = synt_ids_same_tx_number[synt_ids_same_tx_number['n_transcripts'].apply(lambda x: len(x) == 1)]
print(f"Genes with equal transcript numbers across alleles: {len(synt_ids_same_tx_number)}")

# Filter for equal-length transcripts with equal transcript numbers
mask = (adata_isoform_filtered.var["CDS_haplotype_with_longest_annotation"] == "equal_lengths") & \
       (adata_isoform_filtered.var['Synt_id'].isin(synt_ids_same_tx_number.index))

adata_isoform_length_filtered = adata_isoform_filtered[:, mask].copy()

# Plot allelic ratios after filtering
fig = poly.plot_allelic_ratios(
    adata_isoform_length_filtered,
    synteny_category="1hap1_1hap2_1hap3_1hap4_s",
    sample='all',
    ratio_type="both",
    figsize=(6, 3),
    kde=True,
    multimapping_threshold=0.25,
    #save_path="allelic_ratios.svg"
)

In [ ]:
adata_isoform_length_filtered.var[adata_isoform_length_filtered.var['functional_annotation'].str.contains("inhibitor", na=False)]
adata_isoform_length_filtered.var[ adata_isoform_length_filtered.var['synteny_category'].str.contains("3hap1_2hap2_2hap3_3hap4_no_s", na=False)]


In [ ]:
adata_isoform_length_filtered.var['synteny_category'].value_counts()
# print the rows with the 10 rarest syntelog categorys
adata_isoform_length_filtered.var[adata_isoform_length_filtered.var['synteny_category'].isin(adata_isoform_length_filtered.var['synteny_category'].value_counts().nsmallest(100).index)]

# print the syntelog categorys where functional annotation contains "Kutnitz"
#adata_isoform_filtered.var[adata_isoform_filtered.var['functional_annotation'].str.contains("ATPase Family", case=False, na=False)]#['synteny_category'].value_counts()

In [ ]:
# print the counts for a specific synt id
adata_gene_filtered.var[adata_gene_filtered.var['Synt_id'] == "Synt_id_3971"]
adata_gene_filtered.layers["unique_counts"]

df = adata_gene_filtered[:,adata_gene_filtered.var["Synt_id"] == "Synt_id_3149"].layers['em_counts']
pd.DataFrame(data= df, index = adata_gene_filtered[:,adata_gene_filtered.var["Synt_id"] == "Synt_id_3149"].obs.index, columns = adata_gene_filtered[:,adata_gene_filtered.var["Synt_id"] == "Synt_id_3149"].var_names)


Now there are less alleles with ratios around 0 and >0.8.

### A) Cis: Testing for differential allelic expression

We will test within one condition (here: 'leaf'), wether there are genes where alleles have unbalanced expression.

In [ ]:
# select only the genes with equal lengths, low multimapping ratio 

mask = (adata_gene_filtered.var["multimapping_ratio"] < 0.25) & (adata_gene_filtered.var["synteny_category"] == "1hap1_1hap2_1hap3_1hap4_s")& (adata_gene_filtered.var["synt_length_category"] == "uniform_length") & (adata_gene_filtered.var['Synt_id'].isin(synt_ids_same_tx_number.index))

mask = (adata_gene_filtered.var["multimapping_ratio"] < 0.15) & adata_gene_filtered.var["synteny_category"].str.contains("[10]hap1_[10]hap2_[10]hap3_[10]hap4_s") & (adata_gene_filtered.var['Synt_id'].isin(synt_ids_same_tx_number.index)) #& (adata_gene_filtered.var["synt_length_category"] == "uniform_length")

#mask = adata_gene_filtered.var["gene_id"].str.startswith("Soltu.DM.10G02850")
# expression is too low for this gene! maybe in heat I could try ...
adata_gene_length_filter = adata_gene_filtered[:,mask].copy()

# Test for differential allelic ratios withing conditions
cis_results_control = poly.test_allelic_ratios_within_conditions(adata_gene_length_filter, 
                                                                 layer="unique_counts", 
                                                                 test_condition= "control", 
                                                                 inplace=True)
# Get top differential syntelogs
cis_top_results = poly.get_top_differential_syntelogs(cis_results_control, 
                                                      n=45, sort_by='ratio_difference',
                                                      fdr_threshold=0.05, 
                                                      ratio_threshold=0.1)

# save the results to a CSV file
cis_results_control.to_csv("LR_DESIREE_PAPER/ANALYSIS/final_tables_paper/long_cis_results_control.tsv", index=False)

In [ ]:

# reset the plotting parameters to default
plt.rcParams.update(plt.rcParamsDefault)
gene_id = "Soltu.DM.06G029570" 
#Soltu.DM.01G046170
# set dpi for better resolution
cis_results_not_pairwise = cis_results_control
# filter results for specific synt id
cis_results_filtered_synt = cis_results_not_pairwise[cis_results_not_pairwise['gene_id'].str.contains(gene_id)]
ax1 = poly.plot_top_differential_syntelogs_annotated(cis_results_filtered_synt, n = 20, figsize = (14,4),sort_by='ratio_difference')

# filter results for specific synt id
cis_results_filtered_synt = cis_results_not_pairwise[cis_results_not_pairwise['gene_id'].str.contains(gene_id)]
ax2 = poly.plot_top_differential_syntelogs_annotated(cis_results_filtered_synt, n = 20, figsize = (12, 3),sort_by='ratio_difference', plot_type='cpm' )
# ssave as svg
plt.savefig("LR_DESIREE_PAPER/ANALYSIS/final_plots_svg/cytochrome_SV_allelic_ratios_cis_example_Soltu.DM.06G029570.svg", format="svg", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
cis_results_not_pairwise = cis_results_control


In [ ]:
# plot the number of genes per synt id per chromosome
import numpy as np
adata_gene.var['chromosome'] = adata_gene.var['gene_id'].str.split('DM.').str[1].str.split('G').str[0]
# if chromosome is na, extract it from the original gene_id (e.g.,Sotub01g023930.Hap4" --> 1)
if adata_gene.var['chromosome'].isna().any():
    adata_gene.var.loc[adata_gene.var['chromosome'].isna(), 'chromosome'] = adata_gene.var.loc[adata_gene.var['chromosome'].isna(), 'gene_id'].str.extract(r'Sotub(\d+)g\d+')[0]
# chromosomes that contain 'Hap' or 'Un' set to NA
adata_gene.var.loc[adata_gene.var['chromosome'].str.contains('Hap|Un', na=False), 'chromosome'] = np.nan
# count number of syntelogs per chromosome
synt_counts = adata_gene.var.groupby(['chromosome'])['synteny_category'].value_counts().unstack(fill_value=0)

# Group all but top 5 categories into "Other"
top5 = synt_counts.sum().nlargest(9).index
other = synt_counts.columns.difference(top5)
synt_plot = synt_counts[top5].copy()
if len(other) > 0:
    synt_plot["Other"] = synt_counts[other].sum(axis=1)

synt_plot.plot(kind='bar', stacked=True, figsize=(10, 4))
plt.legend(title='Synteny Category', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xlabel('Chromosome')
plt.ylabel('Number of Genes')
plt.title('Number of Genes per Synteny Category by Chromosome')
plt.tight_layout()

In [ ]:
#cis_results_control.to_csv("LR_DESIREE_PAPER/ANALYSIS/ASE_results/long_cis_results_control.tsv", index=False)

In [ ]:
# check for the different values in n_alleles the number of unbalanced synt_ids
# add a column to cis_results_control that indicates whether the synt_id is unbalanced or not
cis_results_control['unbalanced'] = (cis_results_control['FDR'] < 0.05) & (cis_results_control['ratio_difference'].abs() > 0.1)

# group by Synt_id: a Synt_id is unbalanced if any pairwise comparison is unbalanced
synt_status = cis_results_control.groupby('Synt_id').agg(
    unbalanced=('unbalanced', 'any'),
    n_alleles=('n_alleles', 'first')
)

for n_alleles, group in synt_status.groupby("n_alleles"):
    n_unbalanced = group['unbalanced'].sum()
    n_balanced = (~group['unbalanced']).sum()
    print(f"n_alleles: {n_alleles}, unbalanced synt_ids: {n_unbalanced}, balanced synt_ids: {n_balanced}")
    total = n_unbalanced + n_balanced
    if total > 0:
        print(f"Percentage unbalanced: {n_unbalanced / total * 100:.2f}%")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
# filter for syntelogs with 4 alleles and balanced
cis_results_control_4alleles = cis_results_control[cis_results_control['n_alleles'] == 4]

# add chromosome information to cis_results_control from allelic_data_filtered based on gene_id

cis_results_control_4alleles['chromosome'] = cis_results_control_4alleles['gene_id'].str.split('DM.').str[1].str.split('G').str[0]
# if chromosome is na, extract it from the original gene_id (e.g.,Sotub01g023930.Hap4" --> 1)
if cis_results_control_4alleles['chromosome'].isna().any():
    cis_results_control_4alleles.loc[cis_results_control_4alleles['chromosome'].isna(), 'chromosome'] = cis_results_control_4alleles.loc[cis_results_control_4alleles['chromosome'].isna(), 'gene_id'].str.extract(r'Sotub(\d+)g\d+')[0]

cis_results_control_4alleles['chr_hap'] = cis_results_control_4alleles['chromosome'] + "." + cis_results_control_4alleles['allele'].str[-1] # 
# remove the rows with chr_hap that contain 'Hap'
#cis_results_control_4alleles = cis_results_control_4alleles[~cis_results_control_4alleles['chr_hap'].str.contains('Hap')]
cis_results_control_4alleles['chr_hap'].unique()

# if chr_hap contains something other than digit or . or 1,2,3,4, remove it
cis_results_control_4alleles = cis_results_control_4alleles[~cis_results_control_4alleles['chr_hap'].str.contains(r'[^0-9\.1-4]')]
# Your existing code
cis_results_control_4alleles['significant'] = (cis_results_control_4alleles['FDR'] < 0.05) & (cis_results_control_4alleles['ratio_difference'] > 0.1)

# Add a temporary column to track original order
cis_results_control_4alleles['plot_order'] = range(len(cis_results_control_4alleles))

# Create the swarm plot
fig, ax = plt.subplots(figsize=(30, 4))
sns.swarmplot(y=cis_results_control_4alleles['ratios_control_mean'], 
              x=cis_results_control_4alleles['chr_hap'], 
              hue=cis_results_control_4alleles['significant'], 
              size=2,
              ax=ax,
            palette={True: 'red', False: 'grey'},
              # order=['1', '2', '3', '4'])
              order = ['01.1', '01.2', '01.3', '01.4',
         '02.1', '02.2', '02.3', '02.4',
         '03.1', '03.2', '03.3', '03.4',
         '04.1', '04.2', '04.3', '04.4', 
         '05.1', '05.2', '05.3', '05.4',
         '06.1', '06.2', '06.3', '06.4',
         '07.1', '07.2', '07.3', '07.4',
         '08.1', '08.2', '08.3', '08.4',
         '09.1', '09.2', '09.3', '09.4',
         '10.1', '10.2', '10.3', '10.4',
         '11.1', '11.2', '11.3', '11.4',
         '12.1', '12.2', '12.3', '12.4'])


# Extract point positions from the plot
points_data = []
for collection in ax.collections:
    offsets = collection.get_offsets().data
    for x, y in offsets:
        points_data.append({'x': x, 'y': y})

# Match points to data by y-values (within tolerance)
point_positions = {}
for idx, row in cis_results_control_4alleles.iterrows():
    y_val = row['ratios_control_mean']
    allele_idx = ['hap1', 'hap2', 'hap3', 'hap4'].index(row['allele'])
    chr_hap  = ['01.1', '01.2', '01.3', '01.4',
         '02.1', '02.2', '02.3', '02.4',
         '03.1', '03.2', '03.3', '03.4',
         '04.1', '04.2', '04.3', '04.4', 
         '05.1', '05.2', '05.3', '05.4',
         '06.1', '06.2', '06.3', '06.4',
         '07.1', '07.2', '07.3', '07.4',
         '08.1', '08.2', '08.3', '08.4',
         '09.1', '09.2', '09.3', '09.4',
         '10.1', '10.2', '10.3', '10.4',
         '11.1', '11.2', '11.3', '11.4',
         '12.1', '12.2', '12.3', '12.4'].index(row['chr_hap'])

    
    # Find matching point (closest in y-value near the correct x position)
    for point in points_data:
        if abs(point['y'] - y_val) < 0.001 and abs(point['x'] - chr_hap) < 0.5:
            if idx not in point_positions:
                point_positions[idx] = (point['x'], point['y'])
                break

# Connect paired points by Synt_id
for synt_id in cis_results_control_4alleles['Synt_id'].unique():
    subset = cis_results_control_4alleles[cis_results_control_4alleles['Synt_id'] == synt_id]
    if len(subset) >= 2:  # Connect all points with same Synt_id
        indices = subset.index.tolist()
        valid_indices = [i for i in indices if i in point_positions]
        
        if len(valid_indices) >= 2:
            for i in range(len(valid_indices) - 1):
                x_vals = [point_positions[valid_indices[i]][0], point_positions[valid_indices[i+1]][0]]
                y_vals = [point_positions[valid_indices[i]][1], point_positions[valid_indices[i+1]][1]]
                ax.plot(x_vals, y_vals, color='gray', linewidth=0.5, alpha=0.2, zorder=0)

plt.yticks([0.0, 0.15, 0.25, 0.35, 0.5, 1.0])
plt.xticks(ticks=range(48), labels=np.tile(np.arange(1, 5), 12))
plt.axhline(y=0.35, color='r', linestyle='--')
plt.axhline(y=0.15, color='r', linestyle='--')

# Add chromosome labels with group separators below x-axis
from matplotlib import transforms
trans = transforms.blended_transform_factory(ax.transData, ax.transAxes)
n_chrs, n_alleles = 12, 4
chr_centers   = np.array([i * n_alleles + 1.5 for i in range(n_chrs)])
sep_positions = np.array([n_alleles * (i + 1) - 0.5 for i in range(n_chrs - 1)])

for i, pos in enumerate(chr_centers):
    ax.text(pos, -0.25, f'chr{i+1:02d}', transform=trans,
            ha='center', va='bottom', fontsize=14, clip_on=False)

for pos in sep_positions:
    ax.plot([pos, pos], [0, -0.20], transform=trans,
            color='black', lw=0.8, clip_on=False)

plt.tight_layout()
# save as svg
plt.savefig("LR_DESIREE_PAPER/ANALYSIS/final_plots_svg/allelic_ratios_swarm_connected.svg", bbox_inches='tight')
plt.show()

## What are these genes chr04_3? are they close together?
chr06_4?

### Do pairwise test


In [ ]:

# Test for differential allelic ratios withing conditions
cis_results_control = poly.test_allelic_ratios_pairwise(adata_gene_length_filter, 
                                                                 layer="unique_counts", 
                                                                 test_condition= "control", 
                                                                 inplace=True, ratio_diff_cutoff= 0.2)
# Get top differential syntelogs
cis_top_results = poly.get_top_differential_syntelogs(cis_results_control, 
                                                      n=45, sort_by='ratio_difference',
                                                      fdr_threshold=0.05, 
                                                      ratio_threshold=0.2)

In [ ]:
cis_results_control["log2FC"] =cis_results_control.apply(
    lambda row: np.log2(row[f"cpm_allele_{row['allele_i']}_mean"] / row[f"cpm_allele_{row['allele_j']}_mean"])
    if row[f"cpm_allele_{row['allele_j']}_mean"] != 0 else np.nan,
    axis=1
)
# count significant syntelogs with at least logfc of 1 in one of the pairwise comparisons
significant_syntelogs = cis_results_control[(cis_results_control['FDR'] < 0.05) & (abs(cis_results_control['log2FC']) >= 1)].groupby('Synt_id').size()



In [ ]:
cis_results_control["Synt_id"].nunique()
len(significant_syntelogs)

# save cis_results_control as tsv
cis_results_control.to_csv("LR_DESIREE_PAPER/ANALYSIS/final_tables_paper/long_cis_results_control_pairwise.tsv", index=False, sep="\t")

### remove genes from chromosom 8


In [ ]:
cis_results_control = cis_results_control[~cis_results_control["gene_id_i"].str.contains("08G")]

In [ ]:
cis_results_control["log2FC"] = cis_results_control.apply(
    lambda row: np.log2(row[f"cpm_allele_{row['allele_i']}_mean"] / row[f"cpm_allele_{row['allele_j']}_mean"])
    if row[f"cpm_allele_{row['allele_j']}_mean"] != 0 else np.nan,
    axis=1
)
# filter for genes with min log2FC 1 and FDR < 0.05
cis_results_control_filtered = cis_results_control[(cis_results_control['FDR'] < 0.05) & (cis_results_control['log2FC'].abs() > 1)] 
# save
cis_results_control.to_csv("LR_DESIREE_PAPER/ANALYSIS/ASE_results/comparison_DES_ATL/desiree_pairwise_cis_results_all.csv", index=False)

In [ ]:
# check for the different values in n_alleles the number of unbalanced synt_ids
# add a column to cis_results_control that indicates whether the synt_id is unbalanced or not
cis_results_control['unbalanced'] = (cis_results_control['FDR'] < 0.05) & (cis_results_control['ratio_difference'].abs() > 0.2)

# group by Synt_id: a Synt_id is unbalanced if any pairwise comparison is unbalanced
synt_status = cis_results_control.groupby('Synt_id').agg(
    unbalanced=('unbalanced', 'any'),
    n_alleles=('n_alleles', 'first')
)

for n_alleles, group in synt_status.groupby("n_alleles"):
    n_unbalanced = group['unbalanced'].sum()
    n_balanced = (~group['unbalanced']).sum()
    print(f"n_alleles: {n_alleles}, unbalanced synt_ids: {n_unbalanced}, balanced synt_ids: {n_balanced}")
    # add percentage
    total = n_unbalanced + n_balanced
    if total > 0:
        print(f"Percentage unbalanced: {n_unbalanced / total * 100:.2f}%")

In [ ]:
#cis_results_control.to_csv("LR_DESIREE_PAPER/ANALYSIS/ASE_results/Desiree_wo_UTR/cis_results_control_pairwise.csv", index=False)

In [ ]:
# # read in the pariwsie promoter comparison results
# promoter_comparison_file_1 = "SV_whole_genome_desiree/output/syri_parse/hap1_hap2_asm20_optimized_promoter_sv_pairs_f_2000up_500down.tsv"
# promoter_comparison_file_2 = "SV_whole_genome_desiree/output/syri_parse/hap3_hap4_asm20_optimized_promoter_sv_pairs_f_2000up_500down.tsv"
# promoter_comparison_file_3 = "SV_whole_genome_desiree/output/syri_parse/hap1_hap3_asm20_optimized_promoter_sv_pairs_f_2000up_500down.tsv"
# promoter_comparison_file_4 = "SV_whole_genome_desiree/output/syri_parse/hap2_hap4_asm20_optimized_promoter_sv_pairs_f_2000up_500down.tsv"
# promoter_comparison_file_5 = "SV_whole_genome_desiree/output/syri_parse/hap1_hap4_asm20_optimized_promoter_sv_pairs_f_2000up_500down.tsv"
# promoter_comparison_file_6 = "SV_whole_genome_desiree/output/syri_parse/hap2_hap3_asm20_optimized_promoter_sv_pairs_f_2000up_500down.tsv"
# df_promoter_comparison_1 = pd.read_csv(promoter_comparison_file_1, sep="\t", header=0)
# df_promoter_comparison_2 = pd.read_csv(promoter_comparison_file_2, sep="\t", header=0)
# df_promoter_comparison_3 = pd.read_csv(promoter_comparison_file_3, sep="\t", header=0)
# df_promoter_comparison_4 = pd.read_csv(promoter_comparison_file_4, sep="\t", header=0)
# df_promoter_comparison_5 = pd.read_csv(promoter_comparison_file_5, sep="\t", header=0)
# df_promoter_comparison_6 = pd.read_csv(promoter_comparison_file_6, sep="\t", header=0)
# # concatenate the two promoter comparison dataframes
# df_promoter_comparison = pd.concat([df_promoter_comparison_1, df_promoter_comparison_2, df_promoter_comparison_3, df_promoter_comparison_4, df_promoter_comparison_5, df_promoter_comparison_6], ignore_index=True)

# # # subset where ref_gene is in the results_cis
# subset_df = df_promoter_comparison[df_promoter_comparison['ref_gene'].isin(cis_results_control['gene_id_i']) & (df_promoter_comparison['query_gene'].isin(cis_results_control['gene_id_j']))  ]

In [ ]:
# save the subset to save reading next time
#subset_df.to_csv("LR_DESIREE_PAPER/ANALYSIS/notebooks/Polyase_analysis/promoter_comparison_subset.csv", index=False)
import pandas as pd
df_promoter_comparison = pd.read_csv("LR_DESIREE_PAPER/ANALYSIS/notebooks/Polyase_analysis/promoter_comparison_subset.csv")

# remove the SVs where query_dist_tss is greater than 2 kb
# how many are df_promoter_comparison[df_promoter_comparison["query_dist_tss"].abs() <= 2000]
#print(df_promoter_comparison[df_promoter_comparison["query_dist_tss"].abs() > 2000].shape[0])
#df_promoter_comparison = df_promoter_comparison[df_promoter_comparison["query_dist_tss"].abs() <= 3000]

# print svs where query_dist_tss or ref_dist_tss is less than 2000


In [ ]:
df_promoter_comparison["chrB"] = df_promoter_comparison["chrom"].str.rsplit("_", n=1).str[0] + "_" + df_promoter_comparison["query_gene"].str.rsplit("Hap", n=1).str[1]

In [ ]:
# where ref_gene matches gene_id_i in cis_results_control and query_gene matches gene_id_i in cis_results_control, merge the subset_df with cis_results_control on ref_gene and query_gene
merged_df = pd.merge(df_promoter_comparison, cis_results_control, left_on=['ref_gene', 'query_gene'], right_on=['gene_id_i', 'gene_id_j'], how='inner')  
# add a unique identifier for each gene pair
merged_df['gene_pair'] = merged_df['ref_gene'] + "_" + merged_df['query_gene'] 

In [ ]:
# add a uniuque identifier for each SV
merged_df['sv_id_chrom'] = merged_df['chrom'] + "_" + merged_df['sv_id'] + "_" + merged_df['chrB'] 

In [ ]:
print("Number of gene pairs with promoter SVs and tested for allelic imbalance:", merged_df['gene_pair'].nunique())
print("Number of tested gene pairs without promoter SVs:", 
      cis_results_control[~cis_results_control['gene_id_i'].isin(merged_df['ref_gene']) & 
                           ~cis_results_control['gene_id_j'].isin(merged_df['query_gene'])][['gene_id_i', 'gene_id_j']].drop_duplicates().shape[0])

# count gene_pairs 

In [ ]:
import numpy as np


merged_df["log2FC"] = merged_df.apply(
    lambda row: np.log2(row[f"cpm_allele_{row['allele_i']}_mean"] / row[f"cpm_allele_{row['allele_j']}_mean"])
    if row[f"cpm_allele_{row['allele_j']}_mean"] != 0 else np.nan,
    axis=1
)
# remove the rows where log2FC is nan or -inf
merged_df = merged_df[~merged_df["log2FC"].isnull() & ~merged_df["log2FC"].isin([-np.inf, np.inf])].copy()
# where the log2FC is -inf set it to -10 and it it is NaN set it to 10
# merged_df.loc[merged_df['log2FC'] == -np.inf, 'log2FC'] = -10
# merged_df.loc[merged_df['log2FC'].isnull(), 'log2FC'] = 10

mask_unbalanced = (merged_df['FDR'] < 0.01) & (abs(merged_df['log2FC']) > 1)
merged_df['balance_status'] = 'balanced'  # default
merged_df.loc[mask_unbalanced, 'balance_status'] = 'unbalanced'

In [ ]:
# turn around DEL and INS sv_type
merged_df.loc[merged_df['sv_type'] == 'DEL', 'sv_type'] = 'INS_temp'
merged_df.loc[merged_df['sv_type'] == 'INS', 'sv_type'] = 'DEL'
merged_df.loc[merged_df['sv_type'] == 'INS_temp', 'sv_type'] = 'INS'

merged_df_fil =  merged_df[~merged_df["sv_type"].isin(["SYN", "SYNAL", "INVAL", "DUP", "DUPAL", "INV", "CPL", "CPG", "INVDP", "INVTR", 'None', 'INVDPAL', 'INVTRAL', "TRANS", "TRANSAL"])].copy()
lost = merged_df[merged_df["sv_type"].isin(["SYN", "SYNAL", "INVAL", "DUP", "DUPAL", "INV", "CPL", "CPG", "INVDP", "INVTR"])]["ref_gene"].nunique()
total = merged_df["ref_gene"].nunique()
print(f"ref_genes lost: {lost} / {total} ({lost/total*100:.1f}%)")

# sv lengths that are longer than 2000 set to 2000
merged_df_fil.loc[(merged_df_fil["sv_len"] + merged_df_fil["ref_dist_tss"] > 2500), "sv_len"] = 2000 - merged_df_fil.loc[(merged_df_fil["sv_len"] + merged_df_fil["ref_dist_tss"] > 2500), "ref_dist_tss"]
# # remove INS or DEL that are lareger than 2000 bp
#merged_df_fil = merged_df_fil[~((merged_df_fil["sv_type"].isin(["INS", "DEL"])) & (merged_df_fil["sv_len"] > 1000))].copy()
genes_with_full_hdr = merged_df_fil.loc[
    (merged_df_fil["sv_type"] == "HDR") & (merged_df_fil["sv_len"] >=00), "gene_pair"
].unique()
merged_df_fil = merged_df_fil[~merged_df_fil["gene_pair"].isin(genes_with_full_hdr)].copy()
print(f"Removed {len(genes_with_full_hdr)} genes with HDR covering ≥2000 bp of promoter")
merged_grouped_50 = merged_df_fil.groupby("gene_pair").aggregate({
     "gene_pair": "first",
    "ref_start": lambda x: (merged_df_fil.loc[x.index, "sv_len"] > 50).sum(),
    "ref_end": lambda x: (merged_df_fil.loc[x.index, "sv_len"] <= 50).sum(),
    "sv_len":  lambda x: x[merged_df_fil.loc[x.index, "sv_len"] > 50].sum(),
    "balance_status": "first",
    "ref_dist_tss": lambda x: x[merged_df_fil.loc[x.index, "sv_len"] > 50].min(),
    "log2FC": "first",
    "sv_type": lambda x: x[merged_df_fil.loc[x.index, "sv_len"] > 50].loc[merged_df_fil.loc[x.index, "ref_dist_tss"][merged_df_fil.loc[x.index, "sv_len"] > 50].idxmin()] if (merged_df_fil.loc[x.index, "sv_len"] > 50).any() else "None",
    "sv_id": lambda x: x[merged_df_fil.loc[x.index, "sv_len"] > 50].loc[merged_df_fil.loc[x.index, "ref_dist_tss"][merged_df_fil.loc[x.index, "sv_len"] > 50].idxmin()] if (merged_df_fil.loc[x.index, "sv_len"] > 50).any() else "None",
    "sv_id_chrom": lambda x: x[merged_df_fil.loc[x.index, "sv_len"] > 50].loc[merged_df_fil.loc[x.index, "ref_dist_tss"][merged_df_fil.loc[x.index, "sv_len"] > 50].idxmin()] if (merged_df_fil.loc[x.index, "sv_len"] > 50).any() else "None",
    "FDR": "first",
    "functional_annotation": "first"
})  


# rename ref_start to num_sv
merged_grouped_50 = merged_grouped_50.rename(columns={"ref_start": "num_sv"})
merged_grouped_50 = merged_grouped_50.rename(columns={"ref_end": "num_small_sv"})
 
merged_grouped_50["regulation"] = merged_grouped_50.apply(
    lambda row: "up" if (row["log2FC"] > 1 and row["FDR"] < 0.05) 
    else ("down" if (row["log2FC"] < -1 and row["FDR"] < 0.05) 
    else "no_change"), axis=1)


In [ ]:
merged_df_fil["sv_len_short"] = merged_df_fil["sv_len"]
merged_grouped_for_pie_chart = merged_df_fil.groupby("gene_pair").aggregate({
     "gene_pair": "first",
    "ref_start": lambda x: (merged_df_fil.loc[x.index, "sv_len"] > 50).sum(),
    "ref_end": lambda x: (merged_df_fil.loc[x.index, "sv_len"] <= 50).sum(),
    "sv_len":  lambda x: x[merged_df_fil.loc[x.index, "sv_len"] > 50].sum(),
    "balance_status": "first",
    "sv_len_short":  lambda x: x[merged_df_fil.loc[x.index, "sv_len_short"] <= 50].sum(),
    "balance_status": "first",
    "ref_dist_tss": lambda x: x[merged_df_fil.loc[x.index, "sv_len"] > 50].min(),
    "log2FC": "first",
    "sv_type": lambda x: x[merged_df_fil.loc[x.index, "sv_len"] > 50].loc[merged_df_fil.loc[x.index, "ref_dist_tss"][merged_df_fil.loc[x.index, "sv_len"] > 50].idxmin()] if (merged_df_fil.loc[x.index, "sv_len"] > 50).any() else "None",
    "sv_id": lambda x: x[merged_df_fil.loc[x.index, "sv_len"] > 50].loc[merged_df_fil.loc[x.index, "ref_dist_tss"][merged_df_fil.loc[x.index, "sv_len"] > 50].idxmin()] if (merged_df_fil.loc[x.index, "sv_len"] > 50).any() else "None",
    "sv_id_chrom": lambda x: x[merged_df_fil.loc[x.index, "sv_len"] > 50].loc[merged_df_fil.loc[x.index, "ref_dist_tss"][merged_df_fil.loc[x.index, "sv_len"] > 50].idxmin()] if (merged_df_fil.loc[x.index, "sv_len"] > 50).any() else "None",
    "FDR": "first",
    "functional_annotation": "first"
})  


# rename ref_start to num_sv
merged_grouped_for_pie_chart = merged_grouped_for_pie_chart.rename(columns={"ref_start": "num_large_sv"})
merged_grouped_for_pie_chart = merged_grouped_for_pie_chart.rename(columns={"ref_end": "num_small_sv"})
merged_grouped_for_pie_chart = merged_grouped_for_pie_chart.rename(columns={"sv_len": "length_large_sv"})

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# ===================================================
# CONFIGURATION
# ===================================================

PROMOTER_LENGTH = 2500

OUTPUT_PATH = (
    "LR_DESIREE_PAPER/ANALYSIS/final_plots_svg/"
    "SV_bar_charts.svg"
)

df = merged_grouped_for_pie_chart.copy()

# ===================================================
# DERIVED COLUMNS
# ===================================================

df["sv_group"] = df["num_large_sv"].apply(
    lambda x: str(x) if x <= 2 else "3+"
)

df["identity"] = 1 - df["sv_len_short"] / PROMOTER_LENGTH

bin_edges  = [0, 0.9, 0.95, 0.98, 1.001]
bin_labels = ["0–90%", "90–95%", "95–98%", "99–100%"]

mask0 = df["sv_group"] == "0"

df["identity_group"] = None
df.loc[mask0, "identity_group"] = pd.cut(
    df.loc[mask0, "identity"],
    bins=bin_edges,
    labels=bin_labels,
    include_lowest=True,
    right=True,
).astype(str)

df["balance_status"] = df["balance_status"].astype(str)

# ===================================================
# BALANCE COLORS & ORDER
# ===================================================

balance_values = sorted(df["balance_status"].unique())

balance_palette = {
    bal: color for bal, color in zip(balance_values, [
        "#1b9e77", "#d95f02", "#7570b3", "#e7298a", "#66a61e", "#e6ab02",
    ])
}

# ===================================================
# BUILD CROSSTABS → PERCENTAGES
# ===================================================

def to_pct(ct):
    return ct.div(ct.sum(axis=1), axis=0) * 100

ct0_raw = (
    df[mask0]
    .groupby(["identity_group", "balance_status"], observed=True)
    .size()
    .unstack(fill_value=0)
    .reindex(index=bin_labels, columns=balance_values, fill_value=0)
)
ct0 = to_pct(ct0_raw)
ct0_labels = [f"{g}\n(n={ct0_raw.loc[g].sum()})" for g in ct0_raw.index]

sv_order = ["0", "1", "2", "3+"]
sv_order = ["3+", "2", "1", "0"]
ctSV_raw = (
    df
    .groupby(["sv_group", "balance_status"], observed=True)
    .size()
    .unstack(fill_value=0)
    .reindex(index=sv_order, columns=balance_values, fill_value=0)
)
ctSV_raw.index = ["0 SVs", "1 SV", "2 SVs", "3+ SVs"]
ctSV_raw.index = ["3+ SVs", "2 SVs", "1 SV", "0 SVs"]
ctSV = to_pct(ctSV_raw)
ctSV_labels = [f"{g}\n(n={ctSV_raw.loc[g].sum()})" for g in ctSV_raw.index]

# ===================================================
# STYLE
# ===================================================

plt.rcParams.update({
    "font.family":       "sans-serif",
    "font.size":         7,
    "axes.linewidth":    0.5,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "xtick.major.size":  2.5,
    "ytick.major.size":  2.5,
    "xtick.major.width": 0.5,
    "ytick.major.width": 0.5,
    "xtick.bottom":      True,
    "ytick.left":        True
})

#set seaborn style
import seaborn as sns

plt.style.use('seaborn-v0_8-white')

# ===================================================
# PLOT HELPER
# ===================================================

def grouped_bar_ax(ax, ct_pct, ct_raw, xticklabels, xlabel):
    n_groups = len(ct_pct)
    n_bars   = len(balance_values)
    width    = 0.65 / n_bars
    x        = np.arange(n_groups)

    for i, bal in enumerate(balance_values):
        offsets  = x + (i - n_bars / 2 + 0.5) * width
        pct_vals = ct_pct[bal].values
        n_vals   = ct_raw[bal].values
        ax.bar(
            offsets, pct_vals,
            width=width * 0.9,
            color=balance_palette[bal],
            label=bal,
            linewidth=0,
        )

    ax.set_xticks(x)
    ax.set_xticklabels(xticklabels, fontsize=7)
    ax.set_xlabel(xlabel, fontsize=7, labelpad=4)
    ax.set_ylim(0, 100)
    ax.set_yticks([0, 25, 50, 75, 100])
    ax.yaxis.set_major_formatter(mticker.PercentFormatter())
    ax.set_ylabel("Proportion (%)", fontsize=7)
    ax.tick_params(axis="both", labelsize=7, length=2.5)
    ax.grid(axis="y", color="0.90", linewidth=0.4, linestyle="-")
    ax.set_axisbelow(True)

# ===================================================
# FIGURE
# ===================================================

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(2.5, 5), constrained_layout=True)

grouped_bar_ax(ax1, ctSV, ctSV_raw, ctSV_labels, "Number of large SVs")
grouped_bar_ax(ax2, ct0,  ct0_raw,  ct0_labels,  "Promoter identity (0 SVs only)")


for ax, label in zip([ax1, ax2], ["A", "B"]):
    ax.text(-0.10, 1.04, label, transform=ax.transAxes,
            fontsize=9, fontweight="bold", va="bottom")

handles = [plt.Rectangle((0, 0), 1, 1, color=balance_palette[b], linewidth=0)
           for b in balance_values]
fig.legend(
    handles, balance_values,
    ncol=len(balance_values),
    loc="lower center",
    bbox_to_anchor=(0.5, -0.03),
    frameon=False,
    fontsize=7,
    title="Balance status",
    title_fontsize=7,
)

# ===================================================
# SAVE
# ===================================================

fig.savefig(OUTPUT_PATH, bbox_inches="tight", dpi=300)
plt.show()

In [ ]:
print(plt.style.available)

In [ ]:
df.sort_values(["identity"], ascending=False).head(20)

In [ ]:
# for the gene pairs with less than 1 large SV att the length of small variations
small_sv_length_distribution = merged_grouped_for_pie_chart[merged_grouped_for_pie_chart['num_large_sv'] == 0]['sv_len_short'].value_counts().sort_index()
print(small_sv_length_distribution)

In [ ]:
syn_types = {"SYN", "SYNAL", "INVAL", "DUP", "DUPAL", "INV", "CPL", "CPG", "INVDP", "INVTR"}
only_syn = merged_df.groupby("gene_pair")["sv_type"].apply(lambda x: set(x).issubset(syn_types))
print(f"ref_genes with only SYN-like sv_types: {only_syn.sum()}")
only_syn = merged_df.groupby("gene_pair")["sv_type"].apply(lambda x: set(x).issubset(syn_types))
print(only_syn[only_syn].index.tolist())

In [ ]:
for sv_len in [ 50, 60, 70]:
    grouped_df = merged_df_fil.groupby("gene_pair").aggregate({
     "gene_pair": "first",
    "ref_start": lambda x: (merged_df_fil.loc[x.index, "sv_len"] > sv_len).sum()})
    print(f"Gene pairs with more than one SV longer than {sv_len}: {len(grouped_df[grouped_df['ref_start'] > 1])}")
    print(f"Number of gene pairs with exactly 1 SV: {len(grouped_df[grouped_df['ref_start'] == 1])}")
    print(f"Number of gene pairs with no SV: {len(grouped_df[grouped_df['ref_start'] < 1])}")

In [ ]:
914 +  2059 + 2701

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt



# add shape of the points based on sv type
# only plot the ones that have only one sv with length greater than 50 bp
merged_grouped_one_sv = merged_grouped_50[merged_grouped_50["num_sv"] ==1]
# set the point size based on sv length
# add a sv_id_chrom for labeling the points if log2FC is greater than 1 or less than -1



In [ ]:
merged_grouped_one_sv["sv_type"].value_counts()

## Legnth difference for Promotors with one SV longer than 50 bp

In [ ]:
merged_grouped_one_sv["sv_id_chrom"].to_csv("LR_DESIREE_PAPER/ANALYSIS/ASE_results/SV_analysis/one_sv_50_id_chrom.csv", index=False)
TE_classification_file = "SV_whole_genome_desiree/output/TE_analysis/TE_sv50_blast_results_with_header.txt"
df_TE_classification = pd.read_csv(TE_classification_file, sep="\t", header=0)
df_TE_classification["TE_class"] = df_TE_classification["sseqid"].str.split("#").str[1]
df_TE_classification["TE_class_broad"] = df_TE_classification["TE_class"].str.split("/").str[0]

In [ ]:
# take for each qseqid the best hit based on bitscore
df_TE_classification = df_TE_classification.sort_values('bitscore', ascending=False).drop_duplicates('qseqid')
df_TE_classification = df_TE_classification.rename(columns={"qseqid": "sv_id_chrom"})
# merge with the one sv dataframe
merged_grouped_one_sv = pd.merge(merged_grouped_one_sv, df_TE_classification[["sv_id_chrom", "TE_class", "TE_class_broad", "sseqid"]], on="sv_id_chrom", how="left")
# for empty TE_class fill with "None"
merged_grouped_one_sv["TE_class"] = merged_grouped_one_sv["TE_class"].fillna("None")
merged_grouped_one_sv["TE_class_broad"] = merged_grouped_one_sv["TE_class_broad"].fillna("None")


In [ ]:
te_class_counts = merged_grouped_one_sv['TE_class_broad'].value_counts()
# group classes with less than 5% into "Other"
te_class_counts = te_class_counts[te_class_counts > te_class_counts.sum() * 0.05]
te_class_counts['Other'] = (~merged_grouped_one_sv['TE_class_broad'].isin(te_class_counts.index)).sum()

# replace None with no TE
te_class_counts = te_class_counts.rename(index={"None": "no TE"})
palette={"no TE": "#999999", "DNA": "#0072B2", "LINE": "#E69F00", "LTR": "#CC79A7", "SINE": "#56B4E9", "MITE": "#009E73", "Unknown": "#F0E442", "rRNA": "#D55E00", "RC": "#D55E00"},
colors = [te_colors.get(c, "#DDDDDD") for c in te_class_counts.index]

plt.figure(figsize=(4, 4))
plt.pie(te_class_counts, labels=te_class_counts.index, autopct='%1.1f%%', startangle=140, colors=colors)

okabe_ito = ["#999999", "#0072B2", "#E69F00", "#CC79A7", "#56B4E9", "#009E73", "#F0E442", "#D55E00"]

te_class_counts = merged_grouped_one_sv['TE_class'].value_counts()
te_class_counts = te_class_counts[te_class_counts > te_class_counts.sum() * 0.05]
te_class_counts['Other'] = (~merged_grouped_one_sv['TE_class'].isin(te_class_counts.index)).sum()
colors = [okabe_ito[i % len(okabe_ito)] for i in range(len(te_class_counts))]

# save this pie chart
plt.savefig("LR_DESIREE_PAPER/ANALYSIS/final_plots_svg/TE_class_broad_distribution_pie_chart.svg", format="svg")
plt.show()

plt.figure(figsize=(4, 4))
plt.pie(te_class_counts, labels=te_class_counts.index, autopct='%1.1f%%', startangle=140, colors=colors)

# save this pie chart
plt.savefig("LR_DESIREE_PAPER/ANALYSIS/final_plots_svg/TE_class_distribution_pie_chart.svg", format="svg")
plt.show()

In [ ]:
te_class_counts

In [ ]:
te_class_counts

In [ ]:
# where ref_gene matches gene_id_i in cis_results_control and query_gene matches gene_id_i in cis_results_control, merge the subset_df with cis_results_control on ref_gene and query_gene
merged_df = pd.merge(df_promoter_comparison, cis_results_control, left_on=['ref_gene', 'query_gene'], right_on=['gene_id_i', 'gene_id_j'], how='inner')  
# add a unique identifier for each gene pair
merged_df['gene_pair'] = merged_df['ref_gene'] + "_" + merged_df['query_gene'] 
# submet for only n_alleles == 4 to avoid bias from different n_alleles
# how many genes have n_alleles == 4 in the merged_df?
n_alleles_counts = merged_df['n_alleles'].value_counts()

#merged_df = merged_df[merged_df['n_alleles'] == 4]


In [ ]:
merged_grouped_one_sv
# plot ref_dist_tss vs log2FC split by sv_type

# set sytle for plot
sns.set_theme(rc={'figure.figsize':(12,10
                                    )},context="talk", style="white")

merged_grouped_one_sv["TE_class_broad"] = merged_grouped_one_sv["TE_class_broad"].replace("None", "no TE")
p1 = sns.scatterplot(
    merged_grouped_one_sv,
    x=merged_grouped_one_sv["ref_dist_tss"],
    y=merged_grouped_one_sv["log2FC"],
    #hue=merged_grouped_one_sv["balance_status"],
    style=merged_grouped_one_sv["sv_type"],
    markers={"INS": "^", "DEL": "X", "HDR": "."},
    # palette={"balanced": "grey", "unbalanced": "red"},
    # color by TE class broad
    hue=merged_grouped_one_sv["TE_class_broad"],
    palette={"no TE": "#999999", "DNA": "#0072B2", "LINE": "#E69F00", "LTR": "#CC79A7", "SINE": "#56B4E9", "MITE": "#009E73", "Unknown": "#F0E442", "rRNA": "#D55E00", "RC": "#D55E00"},
    alpha=1, sizes = (40, 300))
    # size=merged_grouped_one_sv["sv_len"],
    # sizes=(40, 300))
# turn the x axis around
plt.gca().invert_xaxis()
# change the x axis label to "Distance of SV to TSS (bp)"
plt.xlabel("Distance of SV to TSS (bp)")
plt.ylabel("Log2 Fold Change")
# chagne y to logscale


# move legend outside the plot
plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)
for line in range(0, merged_grouped_one_sv.shape[0]):
    # remove None from the TE class broad labels

    if abs(merged_grouped_one_sv["log2FC"].iloc[line]) > 3:
        # plot non overlapping labels by checking if the distance between the current point and all previously labeled points is greater than a certain threshold (e.g., 0.1 on the y-axis)
        p1.text(merged_grouped_one_sv["ref_dist_tss"].iloc[line]+0.01, merged_grouped_one_sv["log2FC"].iloc[line], 
        merged_grouped_one_sv["TE_class_broad"].iloc[line], horizontalalignment='left', 
        size=15, color='black')


# save as svg
plt.savefig("LR_DESIREE_PAPER/ANALYSIS/final_plots_svg/SV_distance_vs_log2FC_one_sv_50_TE_class_broad.svg", format="svg", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
merged_grouped_50[merged_grouped_50['gene_pair'].str.contains("Soltu.DM.01G005630", case=False, na=False)]

In [ ]:

# print where blaance status is unbalanced and ref_dist_tss is less negative and log2FC is greater than 1
print(merged_grouped_one_sv[(merged_grouped_one_sv["balance_status"] == "unbalanced") & (merged_grouped_one_sv["ref_dist_tss"] > 4) & (merged_grouped_one_sv["log2FC"] > 4)].head(10))

In [ ]:
merged_grouped_one_sv.sort_values("sv_len", ascending=False).head(10)[["gene_pair", "log2FC", "FDR", "sv_type", "sv_len", "TE_class_broad"]]
# save the merged_grouped_one_sv dataframe to csv
merged_grouped_one_sv.to_csv("LR_DESIREE_PAPER/ANALYSIS/final_tables_paper/SV_analysis_50_one_sv.csv", index=False)

### Gene lists export


In [ ]:
# get a list of genes that are "up" and have a "INS" or a "INS" and down
genes_up_ins = merged_grouped_one_sv[(merged_grouped_one_sv["regulation"] == "up") & (merged_grouped_one_sv["sv_type"] == "INS")]["sv_id_chrom"].tolist()
genes_down_del = merged_grouped_one_sv[(merged_grouped_one_sv["regulation"] == "down") & (merged_grouped_one_sv["sv_type"] == "DEL")]["sv_id_chrom"].tolist()

# add lists together and export
genes_up_ins_down_del = genes_up_ins + genes_down_del
genes_up_ins_down_del_df = pd.DataFrame(genes_up_ins_down_del, columns=["sv_id_chrom"])
genes_up_ins_down_del_df.to_csv("LR_DESIREE_PAPER/ANALYSIS/ASE_results/SV_analysis/enrichment/genes_up_ins_down_del.csv", index=False)

# get a list of unchanged INS and DEL
genes_unchanged_ins = merged_grouped_one_sv[(merged_grouped_one_sv["regulation"] == "no_change") & (merged_grouped_one_sv["sv_type"] == "INS")]["sv_id_chrom"].tolist()
genes_unchanged_del = merged_grouped_one_sv[(merged_grouped_one_sv["regulation"] == "no_change") & (merged_grouped_one_sv["sv_type"] == "DEL")]["sv_id_chrom"].tolist()

# add lists together and export
genes_unchanged_ins_del = genes_unchanged_ins + genes_unchanged_del
genes_unchanged_ins_del_df = pd.DataFrame(genes_unchanged_ins_del, columns=["sv_id_chrom"])
genes_unchanged_ins_del_df.to_csv("LR_DESIREE_PAPER/ANALYSIS/ASE_results/SV_analysis/enrichment/genes_unchanged_ins_del.csv", index=False)

In [ ]:
plt.figure(figsize=(3,3))
#sns.countplot(merged_grouped_one_sv, x=merged_grouped_one_sv["regulation"], hue= merged_grouped_one_sv["sv_type"] )
# g = sns.FacetGrid(merged_grouped_one_sv, col="TE_class", sharey=True)
# g.map_dataframe(sns.countplot, x="regulation", hue="sv_type")
# g.add_legend()

sns.displot(merged_grouped_one_sv, x="regulation", hue="TE_class_broad", col="sv_type", multiple="stack", shrink=0.8)
# set ylimit to 10 and -10
#plt.ylim(0, 100)

In [ ]:
merged_grouped_one_sv.to_csv("LR_DESIREE_PAPER/ANALYSIS/ASE_results/pairwise_cisASE_SV_merged_grouped_one_sv.csv")

In [ ]:
merged_grouped_one_sv.sort_values("log2FC", ascending=False).head(2)[["sv_len", "ref_dist_tss", "log2FC", "balance_status", "sv_type", "sv_id"]]

In [ ]:
import pandas as pd

pd.DataFrame({
    "all": merged_grouped_50.value_counts("balance_status"),
    "no_sv_50": merged_grouped_50[merged_grouped_50["num_sv"] == 0].value_counts("balance_status"),
     "one_sv_50": merged_grouped_50[merged_grouped_50["num_sv"] == 1].value_counts("balance_status"),
    "multi_sv_50": merged_grouped_50[merged_grouped_50["num_sv"] > 1].value_counts("balance_status")
}).fillna(0).astype(int)

In [ ]:
# reduce size of plot
sns.set_theme(rc={'figure.figsize':(4,4
                                    )},context="talk")
# make 
sns.displot(merged_grouped_50, hue=merged_grouped_50["balance_status"], x=merged_grouped_50["num_sv"], palette={"balanced": "blue", "unbalanced": "red"}, stat="proportion", common_norm=False, multiple="dodge", aspect=1)
# set the x ticks to be integers
plt.xticks(range(0, merged_grouped_50["num_sv"].max() + 1, 1))
# change the  x axis label
plt.xlabel("Number of SVs (>50 bp)")
plt.ylabel("Proportion of Promotors")
plt.show()

## plot promoter SV

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

PROMOTER_START = -500
PROMOTER_END   = 2000


def _prepare_sv_df(df, gene_name, short_sv_threshold, min_sv_length=0):
    """Filter, compute relative coords from ref_dist_tss, clip to promoter window, and split into long/short SVs."""
    gene_df = df[df["ref_gene"].str.contains(gene_name, na=False)].copy()
    if gene_df.empty:
        raise ValueError(f"No SVs found for gene '{gene_name}'")

    # Use ref_dist_tss directly instead of recomputing from ref_start/ref_end
    gene_df["rel_start"] = gene_df["ref_dist_tss"]
    gene_df["rel_end"]   = gene_df["ref_dist_tss"] + gene_df["sv_len"]

    # Remove synonymous SVs
    gene_df = gene_df[~gene_df["sv_type"].isin(["SYN", "SYNAL", "INVAL"])]
    if gene_df.empty:
        raise ValueError(f"No SVs remaining for gene '{gene_name}' after filtering SYN/SYNAL/INVAL")

    # Keep only SVs that overlap the promoter window
    gene_df = gene_df[
        (gene_df["rel_start"] < PROMOTER_END) &
        (gene_df["rel_end"]   > PROMOTER_START)
    ]
    if gene_df.empty:
        raise ValueError(f"No SVs in promoter region for gene '{gene_name}'")

    # Clip to window for display
    gene_df["rel_start"] = gene_df["rel_start"].clip(lower=PROMOTER_START)
    gene_df["rel_end"]   = gene_df["rel_end"].clip(upper=PROMOTER_END)
    gene_df["sv_width"]  = gene_df["sv_len"]

    # Optional minimum length filter
    if min_sv_length > 0:
        gene_df = gene_df[gene_df["sv_len"] >= min_sv_length]
        if gene_df.empty:
            raise ValueError(
                f"No SVs >= {min_sv_length} bp in promoter region for gene '{gene_name}'"
            )

    long_svs  = gene_df[gene_df["sv_width"] >= short_sv_threshold]
    short_svs = gene_df[gene_df["sv_width"] <  short_sv_threshold]
    return gene_df, long_svs, short_svs


def plot_sv_promoter(
    df,
    gene_name,
    short_sv_threshold=100000,
    min_sv_length=0,
    type_colors=None,
    show_legend=True,
    figsize=(4, None),
    ax=None,
):
    """
    Plot structural variations relative to the promoter region for a specific gene.
    SVs >= short_sv_threshold bp each get their own row; shorter SVs share one row.
    SYN, SYNAL, and INVAL type SVs are excluded.
    Relative coordinates are derived from ref_dist_tss (not re-computed from ref_start).
    SVs are clipped to the [PROMOTER_START, PROMOTER_END] window.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain columns: ref_gene, ref_dist_tss, sv_len, sv_type, sv_id.
    gene_name : str
        Substring to match against ref_gene.
    short_sv_threshold : int
        SVs >= this length get their own row; smaller SVs share one row.
    min_sv_length : int
        Minimum sv_len to include (default 0 = no filter).
    type_colors : dict, optional
        Mapping sv_type -> color. If None, auto-generated from this gene's types.
    show_legend : bool
        Whether to draw a legend on this axes.
    figsize : tuple
        (width, height). Height auto-computed if None.
    ax : matplotlib Axes, optional
        If provided, draw onto this axes.
    """
    gene_df, long_svs, short_svs = _prepare_sv_df(
        df, gene_name, short_sv_threshold, min_sv_length
    )

    n_rows   = len(long_svs) + (1 if len(short_svs) > 0 else 0)
    row_h    = 0.6
    track_h  = 0.3
    gap      = 0.15
    sv_bar_h = row_h * 0.45

    total_height = track_h + gap + n_rows * row_h
    track_y      = total_height - track_h

    if ax is None:
        h = figsize[1] if figsize[1] is not None else max(1.2, 0.5 + n_rows * 0.35)
        fig, ax = plt.subplots(figsize=(figsize[0], h))
    else:
        fig = ax.get_figure()

    # TSS marker
    ax.axvline(0, color="black", lw=1.2, ls="--", alpha=0.7, zorder=3)
    ax.text(0, track_y + track_h + 0.02, "TSS", ha="center", va="bottom",
            fontsize=8, color="black")

    if type_colors is None:
        sv_types    = gene_df["sv_type"].unique()
        type_colors = dict(zip(sv_types, plt.cm.tab10(np.linspace(0, 0.8, len(sv_types)))))

    def draw_sv(row, bar_y):
        rs    = row["rel_start"]
        re    = row["rel_end"]
        color = type_colors.get(row["sv_type"], "grey")
        width = row["sv_width"]
        if width < 1:
            ax.plot([rs, rs], [bar_y, bar_y + sv_bar_h], color=color, lw=2, zorder=4)
            ax.scatter(rs, bar_y + sv_bar_h / 2, marker="D",
                       color=color, edgecolors="black", s=35, lw=0.6, zorder=5)
        else:
            # anchor at rel_start, draw toward rel_end; works correctly on inverted axis
            patch_x = min(rs, re)
            patch_w = abs(re - rs)
            ax.add_patch(mpatches.FancyBboxPatch(
                (patch_x, bar_y), patch_w, sv_bar_h,
                boxstyle="square,pad=0",
                facecolor=color, edgecolor="black", lw=0.7,
                alpha=0.85, zorder=4,
            ))

    label_x = PROMOTER_START - 80   # still numerically left of -500, now visually on the right

    # Draw long SVs — one per row
    for i, (_, row) in enumerate(long_svs.iterrows()):
        row_center = track_y - gap - (i + 0.5) * row_h
        bar_y      = row_center - sv_bar_h / 2
        ax.axhspan(row_center - row_h / 2, row_center + row_h / 2,
                   color="whitesmoke" if i % 2 == 0 else "white", zorder=0)
        draw_sv(row, bar_y)
        ax.text(label_x, row_center, row.get("sv_id", f"SV{i}"),
                ha="right", va="center", fontsize=7)

    # Draw short SVs — all in one shared row
    if len(short_svs) > 0:
        i_row      = len(long_svs)
        row_center = track_y - gap - (i_row + 0.5) * row_h
        bar_y      = row_center - sv_bar_h / 2
        ax.axhspan(row_center - row_h / 2, row_center + row_h / 2,
                   color="whitesmoke" if i_row % 2 == 0 else "white", zorder=0)
        for _, row in short_svs.iterrows():
            draw_sv(row, bar_y)
        ax.text(label_x, row_center, f"short SVs (< {short_sv_threshold} bp)",
                ha="right", va="center", fontsize=7, style="italic")

    if show_legend:
        type_handles = [mpatches.Patch(color=c, label=t) for t, c in type_colors.items()]
        ax.legend(handles=type_handles, fontsize=7, loc="upper right", framealpha=0.8)

    ax.set_xlim(PROMOTER_END, PROMOTER_START)   # inverted: upstream (-500) on the right
    ax.set_ylim(track_y - gap - n_rows * row_h - 0.05, track_y + track_h + 0.15)
    ax.set_xlabel("Position relative to TSS (bp)", fontsize=9)
    ax.set_yticks([])
    ax.set_title(gene_name, fontsize=9)
    ax.spines[["left", "top", "right"]].set_visible(False)

    return fig, ax


def plot_sv_promoter_multi(
    df,
    gene_names,
    short_sv_threshold=100000,
    min_sv_length=0,
    fig_width=4,
    row_unit=0.35,
    suptitle=None,
):
    """
    Plot SVs in the promoter region for multiple genes, stacked vertically
    with a shared x-axis, shared color scheme, and a single figure-level legend.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain columns: ref_gene, ref_dist_tss, sv_len, sv_type, sv_id.
    gene_names : list of str
        Gene name substrings to plot, one panel per gene.
    short_sv_threshold : int
        SVs >= this length get their own row.
    min_sv_length : int
        Minimum sv_len to include (default 0 = no filter).
    fig_width : float
        Figure width in inches.
    row_unit : float
        Height per SV row in inches.
    suptitle : str, optional
        Figure-level title.
    """
    all_types = set()
    heights   = []
    for gene in gene_names:
        try:
            gene_df, long_svs, short_svs = _prepare_sv_df(
                df, gene, short_sv_threshold, min_sv_length
            )
            all_types.update(gene_df["sv_type"].unique())
            n = len(long_svs) + (1 if len(short_svs) > 0 else 0)
        except ValueError:
            n = 1
        heights.append(max(1, n))

    all_types   = sorted(all_types)
    type_colors = dict(zip(all_types, plt.cm.tab10(np.linspace(0, 0.8, len(all_types)))))

    min_h = 0.8
    fig_h = sum(max(min_h, h * row_unit + 0.6) for h in heights)

    fig, axes = plt.subplots(
        len(gene_names), 1,
        figsize=(fig_width, fig_h),
        sharex=True,
        gridspec_kw={"height_ratios": heights, "hspace": 0.4},
    )
    if len(gene_names) == 1:
        axes = [axes]

    for ax, gene in zip(axes, gene_names):
        try:
            plot_sv_promoter(
                df, gene,
                short_sv_threshold=short_sv_threshold,
                min_sv_length=min_sv_length,
                type_colors=type_colors,
                show_legend=False,
                ax=ax,
            )
        except ValueError as e:
            ax.text(0.5, 0.5, str(e), ha="center", va="center",
                    transform=ax.transAxes, fontsize=8, color="grey")
            ax.set_title(gene, fontsize=9)
            ax.set_yticks([])

    for ax in axes:
        ax.set_xlim(PROMOTER_END, PROMOTER_START)   # inverted: upstream (-500) on the right

    for ax in axes[:-1]:
        ax.set_xlabel("")
    axes[-1].set_xlabel("Position relative to TSS (bp)", fontsize=9)

    type_handles = [mpatches.Patch(color=c, label=t) for t, c in type_colors.items()]
    fig.legend(handles=type_handles, fontsize=7, loc="upper right",
               bbox_to_anchor=(1.0, 1.0), framealpha=0.8)

    if suptitle:
        fig.suptitle(suptitle, fontsize=11)

    plt.tight_layout()
    return fig, axes

In [ ]:
fig, axes = plot_sv_promoter_multi(
    merged_df[merged_df["gene_pair"] == "Soltu.DM.06G029570.Hap2_Soltu.DM.06G029570.Hap4"],
    gene_names=["Soltu.DM.06G029570.Hap2"],
    short_sv_threshold=100,
    suptitle="Promoter SVs", min_sv_length=0
)
# save as svg
plt.savefig("LR_DESIREE_PAPER/ANALYSIS/final_plots_svg/LTR_example_cytochrome_Soltu.DM.06G029570.Hap2.svg", format="svg", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plot_sv_promoter_multi(
    merged_df[merged_df["gene_pair"] == "Soltu.DM.06G029570.Hap2_Soltu.DM.06G029570.Hap3"],
    gene_names=["Soltu.DM.06G029570.Hap2"],
    short_sv_threshold=100,
    suptitle="Promoter SVs", min_sv_length=0
)
plt.show()

In [ ]:
gene_name = "Soltu.DM.06G029570.Hap1"

# check filter matches
print(merged_df[merged_df["ref_gene"].str.contains(gene_name, na=False)][
    ["sv_id", "sv_type", "ref_start", "ref_end", "ref_tss", "ref_strand", "ref_dist_tss", "sv_len"]
])

In [ ]:
merged_df["gene_pair"] == "Soltu.DM.01G038630.Hap2__Soltu.DM.01G038630.Hap4"

In [ ]:
# number of balanced and unbalanced genes with SVs in promoter region
# count the number of unique ref_genes in merged_df_fil where balance_status is balanced and unbalanced
num_balanced = merged_df_fil[merged_df_fil["balance_status"] == "balanced"]["ref_gene"].nunique()
num_unbalanced = merged_df_fil[merged_df_fil["balance_status"] == "unbalanced"]["ref_gene"].nunique()
print(f"Number of balanced genes with SVs in promoter: {num_balanced}")
print(f"Number of unbalanced genes with SVs in promoter: {num_unbalanced}")

In [ ]:


# save the results
#cis_results_control.to_csv("LR_DESIREE_PAPER/ANALYSIS/ASE_results/Desiree_wo_UTR/cis_results_control.csv", sep=",")


## Small Variation

In [ ]:
genes_with_full_hdr = merged_df_fil.loc[ (merged_df_fil["sv_len"] >=20), "gene_pair"
].unique()
merged_df_fil = merged_df_fil[~merged_df_fil["gene_pair"].isin(genes_with_full_hdr)].copy()
print(f"Removed {len(genes_with_full_hdr)} genes with HDR covering ≥2000 bp of promoter")
# filter this for only SVs with ref_dist_tss > 0 
merged_df_fil_pos = merged_df_fil[merged_df_fil["ref_dist_tss"] > 0]
merged_grouped_small = merged_df_fil_pos.groupby("gene_pair").aggregate({
    "ref_start": lambda x: (merged_df_fil_pos.loc[x.index, "sv_len"] < 20).sum(),
    "sv_len":  lambda x: x[merged_df_fil_pos.loc[x.index, "sv_len"] < 20].sum(),
    "balance_status": "first",
    "ref_dist_tss": lambda x: x[merged_df_fil_pos.loc[x.index, "sv_len"] < 20].min(),
    "log2FC": "first",
    "sv_type": lambda x: x[merged_df_fil_pos.loc[x.index, "sv_len"] < 20].loc[merged_df_fil_pos.loc[x.index, "ref_dist_tss"][merged_df_fil_pos.loc[x.index, "sv_len"] < 20].idxmin()] if (merged_df_fil_pos.loc[x.index, "sv_len"] < 20  ).any() else "None",
    "sv_id": lambda x: x[merged_df_fil_pos.loc[x.index, "sv_len"] < 20].loc[merged_df_fil_pos.loc[x.index, "ref_dist_tss"][merged_df_fil_pos.loc[x.index, "sv_len"] < 20].idxmin()] if (merged_df_fil_pos.loc[x.index, "sv_len"] < 20).any() else "None",
    "sv_id_chrom": lambda x: x[merged_df_fil_pos.loc[x.index, "sv_len"] < 20].loc[merged_df_fil_pos.loc[x.index, "ref_dist_tss"][merged_df_fil_pos.loc[x.index, "sv_len"] < 20].idxmin()] if (merged_df_fil_pos.loc[x.index, "sv_len"] < 20).any() else "None",
    "FDR": "first"
})  


# rename ref_start to num_sv
merged_grouped_small = merged_grouped_small .rename(columns={"ref_start": "num_sv"})

In [ ]:
merged_grouped_small.value_counts("balance_status")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
plt.subplots(figsize=(16,8))

p1 = sns.scatterplot(
    merged_df_fil,
    x=merged_df_fil["ref_dist_tss"],
    y=merged_df_fil["log2FC"],
    hue=merged_df_fil["balance_status"],
    style=merged_df_fil["sv_type"],
    #markers={"INS": "^", "DEL": "X", "HDR": "."},
    palette={"balanced": "grey", "unbalanced": "red"},
    alpha=0.01,
    size=merged_df_fil["sv_len"],
    sizes=(40, 300),
    linewidth=0.05,
    edgecolor="k"
)
# move legend outside the plot
plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)





In [ ]:

# Get total number of gene pairs in each group for normalization
n_balanced = merged_df_fil[merged_df_fil["balance_status"] == "balanced"]["gene_pair"].nunique()
n_unbalanced = merged_df_fil[merged_df_fil["balance_status"] == "unbalanced"]["gene_pair"].nunique()

# sum over each position the number of SVs 
# subset for balanced and unbalanced and plot the density of SVs by distance from TSS
sv_density_balanced = merged_df_fil[merged_df_fil["balance_status"] == "balanced"].groupby("ref_dist_tss")["sv_id"].count().reset_index(name="sv_count")
sv_density_unbalanced = merged_df_fil[merged_df_fil["balance_status"] == "unbalanced"].groupby("ref_dist_tss")["sv_id"].count().reset_index(name="sv_count")
# sum to intervals of 10 bp
sv_density_balanced["ref_dist_tss"] = (sv_density_balanced["ref_dist_tss"] // 50) * 50
sv_density_unbalanced["ref_dist_tss"] = (sv_density_unbalanced["ref_dist_tss"] // 50) * 50
# sv_density_balanced = sv_density_balanced.groupby("ref_dist_tss")["sv_count"].sum().reset_index()
# sv_density_unbalanced = sv_density_unbalanced.groupby("ref_dist_tss")["sv_count"].sum().reset_index()

# Normalize by total number of gene pairs in each group
sv_density_balanced["sv_count"] = sv_density_balanced["sv_count"] / n_balanced
sv_density_unbalanced["sv_count"] = sv_density_unbalanced["sv_count"] / n_unbalanced

plt.figure(figsize=(10, 4))

sns.lineplot(data=sv_density_unbalanced, x="ref_dist_tss", y="sv_count", color="#0072B2", label="Unbalanced")
sns.lineplot(data=sv_density_balanced, x="ref_dist_tss", y="sv_count", color="grey", label="Balanced")

plt.xlabel("Distance of SNPs/INDELs from TSS (bp)")
plt.ylabel("SNPs per gene pair")
plt.xlim(-500, 2000)
# turn x axis around
plt.gca().invert_xaxis()
plt.savefig("LR_DESIREE_PAPER/ANALYSIS/final_plots_svg/SV_density_by_distance_from_TSS_50bp_intervals.svg", format="svg", dpi=300, bbox_inches='tight')


In [ ]:
# correlatino of number of SNPs with log2FC
plt.figure(figsize=(6, 4))
sns.scatterplot(data=merged_grouped_small, x="num_sv", y="log2FC", hue="balance_status", palette={"balanced": "grey", "unbalanced": "red"}, alpha=0.5)
plt.title("Correlation of Number of SVs with log2FC")
plt.xlabel("Number of SVs in Promoter")
plt.ylabel("log2 Fold Change")
plt.legend(title="Balance Status")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
plt.subplots(figsize=(3,3))
# increase dpi
plt.gcf().set_dpi(200)

p1 = sns.scatterplot(
    merged_grouped_small,
    x=merged_grouped_small["ref_dist_tss"],
    y=merged_grouped_small["log2FC"],
    hue=merged_grouped_small["balance_status"],
    style=merged_grouped_small["sv_type"],
    #markers={"INS": "^", "DEL": "X", "HDR": "."},
    palette={"balanced": "grey", "unbalanced": "red"},
    alpha=0.1,
    #size=merged_grouped_small["sv_len"],
    sizes=(40, 300),
    linewidth=0.05,
    edgecolor="k"
)
# move legend outside the plot
plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)
plt.show()

In [ ]:
# correlatino of number of SNPs with log2FC
plt.figure(figsize=(6, 4))
# add absolute log2FC to the dataframe
merged_grouped_small["abs_log2FC"] = merged_grouped_small["log2FC"].abs()
sns.scatterplot(data=merged_grouped_small, x="num_sv", y="abs_log2FC", hue="balance_status", palette={"balanced": "grey", "unbalanced": "red"}, alpha=0.01)
plt.title("Correlation of Number of SVs with log2FC")
plt.xlabel("Number of SVs in Promoter")
plt.ylabel("log2 Fold Change")
plt.legend(title="Balance Status")
plt.show()

In [ ]:
from statannotations.Annotator import Annotator
plt.subplots(figsize=(3,3))
ax = sns.boxplot(data=merged_grouped_small, x="balance_status", y="num_sv", 
                 palette={"balanced": "blue", "unbalanced": "red"})

annotator = Annotator(ax, [("balanced", "unbalanced")], 
                      data=merged_grouped_small, x="balance_status", y="num_sv")
annotator.configure(test="t-test_ind", text_format="star", loc="outside")
annotator.apply_test()
annotator.annotate()



In [ ]:
annotator.annotate()

In [ ]:
sns.boxplot(data=merged_grouped_small, x="ref_dist_tss", y="balance_status")# palette={"balanced": "blue", "unbalanced": "red"})

In [ ]:
from scipy import stats

balanced = merged_grouped_small[merged_grouped_small["balance_status"] == "balanced"]["ref_dist_tss"]
unbalanced = merged_grouped_small[merged_grouped_small["balance_status"] == "unbalanced"]["ref_dist_tss"]

# Non-parametric (recommended for distance data)
stat, p = stats.mannwhitneyu(balanced, unbalanced, alternative='two-sided')
print(f"Mann-Whitney U: stat={stat:.2f}, p={p:.4f}")

# Or parametric t-test
stat_t, p_t = stats.ttest_ind(balanced, unbalanced)
print(f"t-test: stat={stat_t:.2f}, p={p_t:.4f}")
